# Docking benchmark results (Fig. 2, Figs. S6-S14 and S26)

Run from this directory (`figures/benchmark/`). Reads `benchmark_results.pkl`, which
`build_benchmark_results.py --cohort <common_test_pairs.csv>` builds from the per-pose docking CSVs
(`docking/features`, `docking/results/test`, `unseen_systems/`). Plots are
written to `plots/`. The pickle records the input-availability cohort IDs in
`meta.comparison_cohorts`. Main-test rate panels share the 118-pair common cohort;
missing external outputs count as failures. Affinity panels share finite-score
pairs across all displayed methods and repeats. Unseen comparisons share their
own matched pairs.

| File in `plots/` | Paper figure |
|---|---|
| `success_rate_mbg_apo_docked.pdf` | Fig. 2 |
| `success_rate_full_apo_docked.pdf` | Fig. S6 |
| `success_rate_full_vs_mbg_redocked.pdf` | Fig. S7 |
| `cumulative_rmsd_distribution_apo_docked.pdf` | Fig. S8 |
| `cumulative_rmsd_distribution_redocked.pdf` | Fig. S9 |
| `affinity_vs_docking_score_apo_docked.pdf` | Fig. S10 |
| `affinity_vs_docking_score_redocked.pdf` | Fig. S11 |
| `success_rate_gnina_default_vs_unseen_full_vs_mbg.pdf` | Fig. S12 (re-docked) |
| `success_rate_gpdock_default_vs_unseen_full_vs_mbg.pdf` | Fig. S13 |
| `success_rate_metallodock_default_vs_unseen_full_vs_mbg.pdf` | Fig. S14 |
| `mbg_rmsd_by_metal_vina_train.pdf` | Fig. S26 |


In [1]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.lines import Line2D

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

# Built from the per-pose docking CSVs by build_benchmark_results.py.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PKL = os.path.join(NOTEBOOK_DIR, "benchmark_results.pkl")
with open(PKL, "rb") as f:
    R = pickle.load(f)

In [2]:
PLOT_DIR = os.path.join(NOTEBOOK_DIR, "plots")

TOP_N_POSES = [1, 3, 5, 10, 20, 100]

TOP_N = np.arange(len(TOP_N_POSES))
X_LABELS = [str(n) for n in TOP_N_POSES]
CUTOFFS = np.array(R["rmsd_cutoffs"])
os.makedirs(PLOT_DIR, exist_ok=True)
print("Loaded:", os.path.relpath(PKL, NOTEBOOK_DIR))
print("Scenarios:", list(R["scenarios"].keys()))
COMPARISON_COHORTS = R.get("meta", {}).get("comparison_cohorts", {})
print(
    "Comparison cohorts (pairs):",
    {name: len(pairs) for name, pairs in COMPARISON_COHORTS.items()},
)

cm = 1 / 2.54
figsize_two_subplots = (16 * cm, 8 * cm)
figsize_single = (8 * cm, 8 * cm)
fontsize_x_label = 8
fontsize_y_label = 8
fontsize_x_ticks = 8
fontsize_y_ticks = 8
fontsize_legend = 8
fontsize_titles = 8
TWO_SUBPLOTS_BOTTOM = 0.24
LEGEND_Y = -0.05

METHODS_ALL = ["VINA", "SMINA", "GNINA", "AUTODOCK", "GPDOCK", "METALLODOCK"]
METHODS_AFFINITY = ["VINA", "GNINA", "AUTODOCK", "METALLODOCK"]

SCENARIO_LABELS = {
    "redocked": "Re-docked",
    "apo_docked": "Cross-docked",
}

METHOD_COLORS = {
    "VINA": "#332288",
    "SMINA": "#88CCEE",
    "GNINA": "#44AA99",
    "AUTODOCK": "#d97706",
    "GPDOCK": "#7c3aed",
    "METALLODOCK": "#CC6677",
}

_METHOD_DISPLAY_NAMES = {
    "VINA": "Vina",
    "SMINA": "smina",
    "GNINA": "GNINA",
    "AUTODOCK": "AutoDock GPU",
    "GPDOCK": "GPDOCK",
    "METALLODOCK": "MetalloDock",
}


def _display_method_name(method: str) -> str:
    return _METHOD_DISPLAY_NAMES.get(method, method)


def apply_axis_fonts(ax):
    ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax.tick_params(axis="y", labelsize=fontsize_y_ticks)


def finalize_two_subplot_legend(fig, handles, labels):
    fig.tight_layout()
    fig.subplots_adjust(bottom=TWO_SUBPLOTS_BOTTOM)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=max(1, int(np.ceil(len(labels) / 2))),
        fontsize=fontsize_legend,
        bbox_to_anchor=(0.5, LEGEND_Y),
        frameon=False,
    )

Loaded: benchmark_results.pkl
Scenarios: ['redocked', 'apo_docked']
Comparison cohorts (pairs): {'main_test': 118, 'affinity_redocked': 26, 'affinity_apo_docked': 24, 'GPDOCK_unseen': 139, 'METALLODOCK_unseen': 29}


---
## Success rates — line plots

In [3]:
def _plot_success_rate_ax(ax, sr, methods, key_mean, key_std, ylabel):
    for method in methods:
        d = sr.get(method)
        if d is None or d[key_mean] is None:
            continue
        c = METHOD_COLORS[method]
        label = _display_method_name(method)
        mean = np.array(d[key_mean]) * 100
        std = np.array(d[key_std]) * 100
        ax.plot(
            TOP_N, mean, color=c, linewidth=2, marker="o", markersize=5, label=label
        )
        ax.fill_between(
            TOP_N,
            np.clip(mean - std, 0, 100),
            np.clip(mean + std, 0, 100),
            color=c,
            alpha=0.1,
        )
    ax.set_xticks(TOP_N)
    ax.set_xticklabels(X_LABELS, fontsize=fontsize_x_ticks)
    ax.set_yticks(np.arange(0, 105, 10))
    ax.set_ylim(0, 100)
    ax.set_xlabel("Top N Poses", fontsize=fontsize_x_label)
    ax.set_ylabel(ylabel, fontsize=fontsize_y_label)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    apply_axis_fonts(ax)


def plot_success_rates(scenario):
    sr = R["scenarios"][scenario]["success_rates"]

    if scenario == "apo_docked":
        for key_mean, key_std, ylabel, suffix in [
            ("mbg_mean", "mbg_std", "MBG Success Rate (%)", "mbg"),
            ("full_mean", "full_std", "Whole-Molecule Success Rate (%)", "full"),
        ]:
            fig, ax = plt.subplots(1, 1, figsize=figsize_single)
            _plot_success_rate_ax(ax, sr, METHODS_ALL, key_mean, key_std, ylabel)
            ax.legend(fontsize=fontsize_legend, frameon=False)
            fig.tight_layout()
            path = os.path.join(PLOT_DIR, f"success_rate_{suffix}_{scenario}.pdf")
            fig.savefig(path, bbox_inches="tight")
            plt.show()
    else:
        fig, (ax_mbg, ax_full) = plt.subplots(
            1, 2, figsize=figsize_two_subplots, sharey=False
        )
        _plot_success_rate_ax(
            ax_mbg, sr, METHODS_ALL, "mbg_mean", "mbg_std", "MBG Success Rate (%)"
        )
        _plot_success_rate_ax(
            ax_full,
            sr,
            METHODS_ALL,
            "full_mean",
            "full_std",
            "Whole-Molecule Success Rate (%)",
        )
        handles, labels = ax_full.get_legend_handles_labels()
        finalize_two_subplot_legend(fig, handles, labels)
        path = os.path.join(PLOT_DIR, f"success_rate_full_vs_mbg_{scenario}.pdf")
        fig.savefig(path, bbox_inches="tight")
        plt.show()


for scenario in R["scenarios"]:
    print(f"\n{SCENARIO_LABELS[scenario]}")
    plot_success_rates(scenario)


Re-docked



Cross-docked


---
## Cumulative RMSD distributions

In [4]:
def plot_cumulative_rmsd(scenario):
    cum = R["scenarios"][scenario]["cumulative"]
    fig, (ax_mbg, ax_full) = plt.subplots(
        1, 2, figsize=figsize_two_subplots, sharey=False
    )

    for method in METHODS_ALL:
        d = cum.get(method)
        if d is None:
            continue
        c = METHOD_COLORS[method]
        label = _display_method_name(method)
        for ax, mk, sk in [
            (ax_full, "full_mean", "full_std"),
            (ax_mbg, "mbg_mean", "mbg_std"),
        ]:
            if d[mk] is None:
                continue
            mean, std = np.array(d[mk]), np.array(d[sk])
            ax.plot(CUTOFFS, mean, color=c, linewidth=2, label=label)
            ax.fill_between(
                CUTOFFS,
                np.clip(mean - std, 0, 1),
                np.clip(mean + std, 0, 1),
                color=c,
                alpha=0.1,
            )

    for ax, xlabel in [(ax_mbg, "MBG RMSD Cutoff (Å)"), (ax_full, "RMSD Cutoff (Å)")]:
        ax.set_xlim(0, 5)
        ax.set_ylim(0, 1)
        ax.set_xticks(np.arange(0, 5.1, 1))
        ax.set_yticks(np.arange(0, 1.05, 0.1))
        ax.set_xlabel(xlabel, fontsize=fontsize_x_label)
        ax.set_ylabel(
            "Fraction of Pairs with Top-1 RMSD ≤ Cutoff", fontsize=fontsize_y_label
        )
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        apply_axis_fonts(ax)

    handles, labels = ax_full.get_legend_handles_labels()
    finalize_two_subplot_legend(fig, handles, labels)
    path = os.path.join(PLOT_DIR, f"cumulative_rmsd_distribution_{scenario}.pdf")
    fig.savefig(path, bbox_inches="tight")
    plt.show()


for scenario in R["scenarios"]:
    print(f"\n{SCENARIO_LABELS[scenario]}")
    plot_cumulative_rmsd(scenario)


Re-docked



Cross-docked


---
## Leakage analyses

In [5]:
def _extract_method_from_series_name(name):
    name_upper = name.upper()
    for method in ["GNINA", "SMINA", "VINA", "AUTODOCK", "GPDOCK", "METALLODOCK"]:
        if method in name_upper:
            return method
    return None


def _get_ls_from_name(name, fallback="-"):
    name_lower = name.lower()
    if "no cluster" in name_lower:
        return ":"
    elif "no pdb" in name_lower:
        return "--"
    return fallback


def _render_leakage_legend(fig, method_keys, ls_labels):
    """Two titled legends below the figure: Docking tool (left) | Evaluated on (right)."""
    method_h = [
        Line2D([0], [0], color=METHOD_COLORS[m], linewidth=2) for m in method_keys
    ]
    method_l = [_display_method_name(m) for m in method_keys]
    ls_h = [
        Line2D([0], [0], color="grey", linewidth=1.5, linestyle=ls) for ls in ls_labels
    ]
    ls_l = list(ls_labels.values())

    fig.legend(
        method_h,
        method_l,
        title="Docking tool",
        loc="lower center",
        bbox_to_anchor=(0.3, 0.0),
        fontsize=fontsize_legend,
        title_fontsize=fontsize_legend,
        frameon=False,
    )
    fig.legend(
        ls_h,
        ls_l,
        title="Evaluated on:",
        loc="lower center",
        bbox_to_anchor=(0.72, 0.0),
        fontsize=fontsize_legend,
        title_fontsize=fontsize_legend,
        frameon=False,
    )


def plot_unseen_two(series_data, out_name, method_keys, ls_labels):
    fig, (ax_mbg, ax_full) = plt.subplots(
        1, 2, figsize=figsize_two_subplots, sharey=False
    )
    for s in series_data:
        if s["full_mean"] is None and s["mbg1_mean"] is None:
            continue
        method = _extract_method_from_series_name(s["name"])
        color = METHOD_COLORS.get(method, s["color"]) if method else s["color"]
        ls = _get_ls_from_name(s["name"], fallback=s["ls"])
        kw = dict(color=color, linewidth=2, marker="o", markersize=5, linestyle=ls)
        for ax, mk, sk in [
            (ax_full, "full_mean", "full_std"),
            (ax_mbg, "mbg1_mean", "mbg1_std"),
        ]:
            if s[mk] is None:
                continue
            mean = np.array(s[mk]) * 100
            std = np.array(s[sk]) * 100
            ax.plot(TOP_N, mean, **kw)
            ax.fill_between(
                TOP_N,
                np.clip(mean - std, 0, 100),
                np.clip(mean + std, 0, 100),
                color=color,
                alpha=0.1,
            )

    for ax, ylabel in [
        (ax_mbg, "MBG Success Rate (%)"),
        (ax_full, "Whole-Molecule Success Rate (%)"),
    ]:
        ax.set_xticks(TOP_N)
        ax.set_xticklabels(X_LABELS, fontsize=fontsize_x_ticks)
        ax.set_yticks(np.arange(0, 105, 10))
        ax.set_ylim(0, 100)
        ax.set_xlabel("Top N Poses", fontsize=fontsize_x_label)
        ax.set_ylabel(ylabel, fontsize=fontsize_y_label)
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        apply_axis_fonts(ax)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.38)
    _render_leakage_legend(fig, method_keys, ls_labels)
    path = os.path.join(PLOT_DIR, out_name)
    fig.savefig(path, bbox_inches="tight")
    plt.show()


def _all_test_series(method):
    """Re-docking success of ``method`` on the main test pairs, drawn solid."""
    d = R["scenarios"]["redocked"]["success_rates"][method]
    return {
        "name": f"{method} (all test)",
        "color": METHOD_COLORS[method],
        "ls": "-",
        "full_mean": d["full_mean"],
        "full_std": d["full_std"],
        "mbg1_mean": d["mbg_mean"],
        "mbg1_std": d["mbg_std"],
    }


_GNINA_LS = {"-": "All test PL systems", "--": "Unseen PDBs", ":": "Unseen PL clusters"}
_CLUSTER_LS = {"-": "All test PL systems", "--": "Unseen PL clusters"}

plot_unseen_two(
    R["gnina_unseen"],
    "success_rate_gnina_default_vs_unseen_full_vs_mbg.pdf",
    method_keys=["GNINA", "VINA", "SMINA"],
    ls_labels=_GNINA_LS,
)
plot_unseen_two(
    [_all_test_series("GPDOCK")] + R["gpdock_unseen"],
    "success_rate_gpdock_default_vs_unseen_full_vs_mbg.pdf",
    method_keys=["GPDOCK", "VINA", "SMINA"],
    ls_labels=_CLUSTER_LS,
)
plot_unseen_two(
    [_all_test_series("METALLODOCK")] + R["metallodock_unseen"],
    "success_rate_metallodock_default_vs_unseen_full_vs_mbg.pdf",
    method_keys=["METALLODOCK", "VINA", "SMINA"],
    ls_labels=_CLUSTER_LS,
)

---
## Binding affinity vs docking score

In [6]:
def plot_affinity(scenario):
    aff = R["scenarios"][scenario]["affinity"]
    if not aff:
        print(f"No affinity data for {scenario}")
        return

    ncols = 2
    nrows = int(np.ceil(len(METHODS_AFFINITY) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(16 * cm, 16 * cm), constrained_layout=True
    )
    axes = np.array(axes).reshape(-1)

    for ax, method in zip(axes[: len(METHODS_AFFINITY)], METHODS_AFFINITY):
        d = aff.get(method)
        if d is None:
            ax.set_visible(False)
            continue
        scores = np.array(d["scores"])
        affins = np.array(d["affinities"])
        c = METHOD_COLORS[method]
        sns.regplot(
            x=scores,
            y=affins,
            ax=ax,
            color=c,
            scatter_kws={"alpha": 0.6, "s": 25, "zorder": 3},
            line_kws={"linewidth": 1.5},
            ci=95,
        )
        # Free energies (lower is better) are plotted negated, as in the pickle.
        if method in ("VINA", "AUTODOCK"):
            ax.set_xlabel("Neg. Docking Score", fontsize=fontsize_x_label)
        elif method == "METALLODOCK":
            ax.set_xlabel("MDN Score", fontsize=fontsize_x_label)
        else:
            ax.set_xlabel("Predicted Binding Affinity", fontsize=fontsize_x_label)
        ax.set_title(
            f"{_display_method_name(method)} (n={d['n']})", fontsize=fontsize_titles
        )
        ax.grid(True, linestyle="--", alpha=0.4)
        apply_axis_fonts(ax)
        ax.set_box_aspect(1)
        textstr = (
            f"r = {d['pearson_r']:.2f}, p = {d['pearson_p']:.1e}\n"
            f"\u03c1 = {d['spearman_r']:.2f}, p = {d['spearman_p']:.1e}"
        )
        props = dict(
            boxstyle="round", facecolor="white", alpha=0.7, edgecolor="lightgrey"
        )
        ax.text(
            0.05,
            0.95,
            textstr,
            transform=ax.transAxes,
            fontsize=fontsize_legend,
            verticalalignment="top",
            bbox=props,
        )

    for i, ax in enumerate(axes[: len(METHODS_AFFINITY)]):
        if i % ncols == 0:
            ax.set_ylabel(
                r"Binding Affinity [$-\log(K_D/K_I/\mathrm{IC}_{50})$]",
                fontsize=fontsize_x_label,
            )

    path = os.path.join(PLOT_DIR, f"affinity_vs_docking_score_{scenario}.pdf")
    fig.savefig(path, bbox_inches="tight")
    plt.show()


for scenario in R["scenarios"]:
    print(f"\n{SCENARIO_LABELS[scenario]}")
    plot_affinity(scenario)


Re-docked



Cross-docked


---
## MBG RMSD by metal identity

In [7]:
import pandas as pd
from matplotlib.colors import ListedColormap, to_rgba
from scipy import stats

mbm = R["mbg_rmsd_by_metal"]
top_metals = mbm["top_metals"][:6]
metal_rmsds = mbm["metal_rmsds"]

# pairwise matrix
n = len(top_metals)
pmat = np.full((n, n), np.nan)
for i, mi in enumerate(top_metals):
    for j, mj in enumerate(top_metals):
        if i != j:
            _, p = stats.mannwhitneyu(
                metal_rmsds[mi], metal_rmsds[mj], alternative="less"
            )
            pmat[i, j] = p

grey_cell = "#cccccc"
vina_sig = to_rgba(METHOD_COLORS["VINA"], alpha=0.8)
cmap_bin = ListedColormap([grey_cell, vina_sig])
cmat = np.zeros((n, n))
annot_mat = np.full((n, n), "", dtype=object)
for i in range(n):
    for j in range(n):
        if i != j:
            cmat[i, j] = 1.0 if pmat[i, j] < 0.05 else 0.0
            p_ij = pmat[i, j]
            annot_mat[i, j] = f"{p_ij:.1e}" if p_ij < 0.001 else f"{p_ij:.3f}"

rows = [{"Metal": m, "RMSD": v} for m in top_metals for v in metal_rmsds[m]]
df_plot = pd.DataFrame(rows)

fig, (ax_vio, ax_heat) = plt.subplots(
    1,
    2,
    figsize=(16 * cm, 8 * cm),
    gridspec_kw={"width_ratios": [1.5, 1]},
)
vina_color = METHOD_COLORS["VINA"]

# panel a: violin
sns.violinplot(
    data=df_plot,
    x="Metal",
    y="RMSD",
    order=top_metals,
    color=vina_color,
    inner="box",
    cut=0,
    ax=ax_vio,
    alpha=0.8,
)
tick_labels = [f"{m}\nn={len(metal_rmsds[m]):,}" for m in top_metals]
ax_vio.set_xticks(range(len(top_metals)))
ax_vio.set_xticklabels(tick_labels, fontsize=fontsize_x_ticks)
ax_vio.set_xlabel("")
ax_vio.set_ylabel("MBG RMSD (\u00c5)", fontsize=fontsize_y_label)
ax_vio.set_xlim(-0.6, len(top_metals) - 0.4)
ax_vio.set_ylim(bottom=0)
ax_vio.grid(axis="y", linestyle="--", alpha=0.4)
ax_vio.set_title("a) Training MBG RMSDs", fontsize=fontsize_titles, loc="left")
apply_axis_fonts(ax_vio)

# panel b: pairwise MWU heatmap
sns.heatmap(
    cmat,
    annot=annot_mat,
    fmt="",
    xticklabels=top_metals,
    yticklabels=top_metals,
    cmap=cmap_bin,
    vmin=0,
    vmax=1,
    linewidths=0.8,
    linecolor="black",
    cbar=False,
    annot_kws={"size": 5.5, "color": "white"},
    ax=ax_heat,
)
ax_heat.set_title("b) Significant Differences", fontsize=fontsize_titles, loc="left")
ax_heat.set_xlabel("Compared against", fontsize=fontsize_x_label)
ax_heat.set_ylabel("Tested as lower", fontsize=fontsize_y_label)
ax_heat.tick_params(labelsize=fontsize_x_ticks)

fig.tight_layout()
path = os.path.join(PLOT_DIR, "mbg_rmsd_by_metal_vina_train.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()